In [1]:
import json
from pathlib import Path

import pandas as pd
import plotly.express as px
from transformers import PreTrainedTokenizer

from pivotal_tokens.constants import get_data_dir, get_artifacts_dir
from pivotal_tokens.hf.loading import  load_tokenizer
from pivotal_tokens.repo import SampleRepo, DictRepo


EXP_DIR = get_data_dir() / 'experiments' / 'exp2.1-sps-tokens' / 'exp2.1.2-qwen3-1.7b-sps-tokens-small-prob-threshold'
# EXP_DIR = get_data_dir() / 'experiments' / 'exp2.1-sps-tokens' / 'exp2.1.3-qwen3-1.7b-sps-tokens-small-prob-threshold'
# EXP_DIR = get_data_dir() / 'experiments' / 'exp2.1-sps-tokens' / 'exp2.1.4-qwen3-1.7b-sps-tokens-small-prob-threshold'
REPO_DIR = EXP_DIR / 'data' / 'repo'
PIVOTAL_TOKENS_FILE = EXP_DIR / 'data' / 'pivotal_tokens.csv'

TRACES_FILE = get_artifacts_dir() / 'exp1_1_qwen3_1_7b_traces.csv'

QWEN3_1_7B_MODEL_ID = 'Qwen/Qwen3-1.7B'

In [2]:
tokenizer = load_tokenizer(QWEN3_1_7B_MODEL_ID)

In [3]:
base_repo = DictRepo(dirpath=REPO_DIR)

In [4]:
traces_df = pd.read_csv(TRACES_FILE)
traces_df[:2]

,id,query,ground_truth,trace
0,5a8b57f25542995d1e6f1371,Were Scott Derrickson and Ed Wood of the same ...,yes,"<think>\nOkay, let's see. The question is whet..."
1,5a8db19d5542994ba4e3dd00,Are Local H and For Against both from the Unit...,yes,"<think>\nOkay, let's see. The question is aski..."


In [5]:
tokens_df = pd.read_csv(PIVOTAL_TOKENS_FILE)
tokens_df['token_ids'] = tokens_df['token_ids'].apply(json.loads)
tokens_df[:2]

,sample_id,span_id,token_ids,span_text,prob_before,prob_after,prob_delta,is_pivotal,pivotal_context,metadata
0,5a8db19d5542994ba4e3dd00,5cac3872-e49d-11f0-93c4-08bfb8a7cb42,[220],,0.58,0.64,0.06,True,<|im_start|>system\nAnswer the following quest...,"{'id': '5a8db19d5542994ba4e3dd00', 'question':..."
1,5a8db19d5542994ba4e3dd00,5caca596-e49d-11f0-93c4-08bfb8a7cb42,[2704],sure,0.62,0.54,-0.08,True,<|im_start|>system\nAnswer the following quest...,"{'id': '5a8db19d5542994ba4e3dd00', 'question':..."


In [6]:
df = pd.merge(traces_df, tokens_df, left_on='id', right_on='sample_id', how='inner')
df[:2]

,id,query,ground_truth,trace,sample_id,span_id,token_ids,span_text,prob_before,prob_after,prob_delta,is_pivotal,pivotal_context,metadata
0,5a8db19d5542994ba4e3dd00,Are Local H and For Against both from the Unit...,yes,"<think>\nOkay, let's see. The question is aski...",5a8db19d5542994ba4e3dd00,5cac3872-e49d-11f0-93c4-08bfb8a7cb42,[220],,0.58,0.64,0.06,True,<|im_start|>system\nAnswer the following quest...,"{'id': '5a8db19d5542994ba4e3dd00', 'question':..."
1,5a8db19d5542994ba4e3dd00,Are Local H and For Against both from the Unit...,yes,"<think>\nOkay, let's see. The question is aski...",5a8db19d5542994ba4e3dd00,5caca596-e49d-11f0-93c4-08bfb8a7cb42,[2704],sure,0.62,0.54,-0.08,True,<|im_start|>system\nAnswer the following quest...,"{'id': '5a8db19d5542994ba4e3dd00', 'question':..."


In [7]:
def get_thinking_trace_from_completion(completion: str) -> str:
    prefix_split_str = '<|im_start|>assistant\n'
    completion_with_suffix = completion.split(prefix_split_str)[-1].strip()

    suffix_split_str = '<|im_end|>'
    completion = completion_with_suffix.split(suffix_split_str)[0].strip()
    return completion


def get_token_length(text: str, tokenizer: PreTrainedTokenizer) -> int:
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    return len(token_ids)

In [8]:
df['partial_trace'] = df['pivotal_context'].apply(get_thinking_trace_from_completion)
df['partial_trace_token_length'] = df['partial_trace'].apply(lambda x: get_token_length(x, tokenizer))
df['trace_token_length'] = df['trace'].apply(lambda x: get_token_length(x, tokenizer))
df['pivotal_token_relative_position'] = (df['partial_trace_token_length'] + 1) / df['trace_token_length']

assert (df['partial_trace_token_length'] < df['trace_token_length']).mean() == 1.0

In [9]:
def get_init_success_prob(sample_id: str, base_repo: DictRepo) -> float | None:
    repo = SampleRepo(base_repo=base_repo, sample_id=sample_id)
    subdivisions = repo.list(path="subdivisions")

    init_prob = None
    for subdiv in subdivisions:
        data = repo.load(path="subdivisions", key=subdiv)
        if len(data['prefix']) == 0 and \
                data['full_seq'].startswith('<think>') and \
                data['full_seq'].endswith('</think>'):
            init_prob = data['prob_before']
            break
    
    return init_prob

In [10]:
df['prob_init'] = df['sample_id'].apply(lambda x: get_init_success_prob(x, base_repo))

In [11]:
df['prob_before_normalized'] = df['prob_before'] / df['prob_init']
df['prob_after_normalized'] = df['prob_after'] / df['prob_init']
df['prob_delta_normalized'] = df['prob_delta'] / df['prob_init']

In [12]:
df['is_positive'] = df['prob_delta'] >= 0

In [13]:
df = df[df['is_pivotal'] == True].copy()

* * *

In [14]:
print(f"Number of pivotal tokens: {len(df)}")

df['is_positive'].value_counts()

Number of pivotal tokens: 3377


is_positive
True     1928
False    1449
Name: count, dtype: int64

In [15]:
df = df[df['is_positive'] == True].copy()

In [16]:
counts = df.groupby('sample_id', as_index=False).agg({'span_id': 'nunique'})
px.histogram(counts, x='span_id', nbins=20, title='Number of Positive Pivotal Tokens per Sample').show()

In [17]:
# Check relation between pivotal token position and trace length
fig = px.scatter(df,
                x='trace_token_length',
                y='partial_trace_token_length',
                hover_data=['sample_id', 'prob_before', 'prob_after', 'prob_init', 'span_text'],
                title='Partial Trace Token Length vs Full Trace Token Length')
fig.show()

In [18]:
# Check if longer traces correlate with lower initial success probability
fig = px.scatter(df,
                 x='trace_token_length' ,
                 y='prob_init',
                 hover_data=['sample_id', 'prob_before', 'prob_after'],
                 title='Initial Success Probability vs Full Trace Token Length'
                 )
fig.show()

In [19]:
# Check relation between pivotal token position and its impact on success probability
fig = px.scatter(df,
                 x='partial_trace_token_length' ,
                 y='prob_delta',
                 hover_data=['sample_id', 'prob_before', 'prob_after', 'prob_init', 'span_text'],
                 title='Pivotal Token Position vs Impact on Success Probability'
                 )
fig.show()


fig = px.scatter(df,
                 x='pivotal_token_relative_position' ,
                 y='prob_delta',
                 hover_data=['sample_id', 'prob_before', 'prob_after', 'prob_init', 'span_text'],
                 title='Pivotal Token Position vs Impact on Success Probability'
                 )
fig.show()

In [20]:
# Check relation between pivotal token position and its normalized impact on success probability
fig = px.scatter(df,
                 x='partial_trace_token_length' ,
                 y='prob_delta_normalized',
                 hover_data=['sample_id', 'prob_before', 'prob_after', 'prob_init', 'span_text'],
                 title='Pivotal Token Position vs Normalized Impact on Success Probability'
                 )
fig.show()


fig = px.scatter(df,
                 x='pivotal_token_relative_position' ,
                 y='prob_delta_normalized',
                 hover_data=['sample_id', 'prob_before', 'prob_after', 'prob_init', 'span_text'],
                 title='Pivotal Token Position vs Normalized Impact on Success Probability'
                 )
fig.show()

In [21]:
df[df['partial_trace_token_length'] == 1]

,id,query,ground_truth,trace,sample_id,span_id,token_ids,span_text,prob_before,prob_after,...,metadata,partial_trace,partial_trace_token_length,trace_token_length,pivotal_token_relative_position,prob_init,prob_before_normalized,prob_after_normalized,prob_delta_normalized,is_positive
167,5ab41d855542996a3a969f4b,Is the Pakistan fast bowler who joined the Ken...,right-hand,"<think>\nOkay, let's see. The question is abou...",5ab41d855542996a3a969f4b,765a3980-e4c6-11f0-93c4-08bfb8a7cb42,[32313],Okay,0.24,0.66,...,"{'id': '5ab41d855542996a3a969f4b', 'question':...",<think>,1,231,0.008658,0.22,1.090909,3.000000,1.909091,True
204,5ab6369655429953192ad2a1,Which Genus has more species Eucryphia or Lepi...,Eucryphia,"<think>\nOkay, let's see. The question is aski...",5ab6369655429953192ad2a1,2193eb2e-e4cd-11f0-93c4-08bfb8a7cb42,[32313],Okay,0.22,0.42,...,"{'id': '5ab6369655429953192ad2a1', 'question':...",<think>,1,301,0.006645,0.20,1.100000,2.100000,1.000000,True
637,5a84d80c5542992a431d1ab5,Which building is used for more different uses...,MiMA,"<think>\nOkay, let's see. The question is aski...",5a84d80c5542992a431d1ab5,41d58886-e511-11f0-93c4-08bfb8a7cb42,[198],\n,0.52,0.62,...,"{'id': '5a84d80c5542992a431d1ab5', 'question':...",<think>,1,452,0.004425,0.56,0.928571,1.107143,0.178571,True
638,5a84d80c5542992a431d1ab5,Which building is used for more different uses...,MiMA,"<think>\nOkay, let's see. The question is aski...",5a84d80c5542992a431d1ab5,41d5d08e-e511-11f0-93c4-08bfb8a7cb42,[32313],Okay,0.62,0.76,...,"{'id': '5a84d80c5542992a431d1ab5', 'question':...",<think>,1,452,0.004425,0.56,1.107143,1.357143,0.250000,True
726,5a7cdcbf55429907fabef04f,"Which film came out first, Still Kicking: Six ...",Still Kicking: Six Artistic Women of Project A...,"<think>\nOkay, let's see. The user is asking w...",5a7cdcbf55429907fabef04f,9a93ed08-e522-11f0-93c4-08bfb8a7cb42,[32313],Okay,0.44,0.52,...,"{'id': '5a7cdcbf55429907fabef04f', 'question':...",<think>,1,307,0.006515,0.42,1.047619,1.238095,0.190476,True
843,5a7af1a955429931da12c98f,Which Karakoram mountain is higher: Distaghil ...,Distaghil Sar,"<think>\nOkay, let's see. The user is asking w...",5a7af1a955429931da12c98f,86158baa-e538-11f0-93c4-08bfb8a7cb42,[32313],Okay,0.44,0.52,...,"{'id': '5a7af1a955429931da12c98f', 'question':...",<think>,1,213,0.009390,0.46,0.956522,1.130435,0.173913,True
943,5ae3790f5542992e3233c415,Are Leopold Lummerstorfer and Valeria Sarmient...,no,"<think>\nOkay, let's see. The question is whet...",5ae3790f5542992e3233c415,ca532582-e544-11f0-93c4-08bfb8a7cb42,[198],\n,0.22,0.28,...,"{'id': '5ae3790f5542992e3233c415', 'question':...",<think>,1,312,0.006410,0.34,0.647059,0.823529,0.176471,True
1264,5a7e14505542995ed0d16685,"Which canal is located farther east, Champlain...",Champlain Canal,"<think>\nOkay, let's see. The question is aski...",5a7e14505542995ed0d16685,3256122e-e587-11f0-93c4-08bfb8a7cb42,[198],\n,0.52,0.70,...,"{'id': '5a7e14505542995ed0d16685', 'question':...",<think>,1,276,0.007246,0.54,0.962963,1.296296,0.333333,True
1360,5a74feb75542996c70cfae6d,"Which man, Ibn Tufail or Ali Qushji, is best k...",Ali Qushji,"<think>\nOkay, let's tackle this question. The...",5a74feb75542996c70cfae6d,d9ac33de-e58f-11f0-93c4-08bfb8a7cb42,[198],\n,0.62,0.98,...,"{'id': '5a74feb75542996c70cfae6d', 'question':...",<think>,1,370,0.005405,0.66,0.939394,1.484848,0.545455,True
1495,5a81f49c5542995ce29dcc9a,The physicist that saw a deficiency in Newton'...,1921,"<think>\nOkay, so the question is about a phys...",5a81f49c5542995ce29dcc9a,80019ebc-e5b4-11f0-93c4-08bfb8a7cb42,[32313],Okay,0.56,0.62,...,"{'id': '5a81f49c5542995ce29dcc9a', 'question':...",<think>,1,543,0.003683,0.52,1.076923,1.192308,0.115385,True


In [22]:
# TODO:
# 1. Make a list of what I've checked before for the pivotal tokens analysis
# 2. Reproduce all the plots and tables, analyze results, make conclusions. Compare to previous conclusions.
# 3. Prepare random tokens to replace pivotal tokens.
# 4. Prepare script to run the experiments with replaced tokens.
# 5. Run the experiments. 